# 📽️ PLAYWRIGHT - Model Inference

This notebook provides:
1. Load the trained model
2. Complete inference pipeline
3. Export model for production use
4. API-ready inference class

---
**Runtime:** ~5 minutes

## Setup

In [ ]:
!pip install transformers torch -q

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import re
import json
import torch
import torch.nn as nn
from typing import List, Dict, Any, Optional
from transformers import DistilBertTokenizer, DistilBertModel, DistilBertPreTrainedModel, DistilBertConfig

# Paths
DATA_PATH = "/content/drive/MyDrive/playwright_data"
MODEL_PATH = f"{DATA_PATH}/model/final_model"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load Model

In [ ]:
# Load label mappings
with open(f"{MODEL_PATH}/label_mappings.json", 'r') as f:
    mappings = json.load(f)

LABEL_TO_ID = mappings['LABEL_TO_ID']
ID_TO_LABEL = {int(k): v for k, v in mappings['ID_TO_LABEL'].items()}
MOOD_TO_ID = mappings['MOOD_TO_ID']
ID_TO_MOOD = {int(k): v for k, v in mappings['ID_TO_MOOD'].items()}
CAMERA_TO_ID = mappings['CAMERA_TO_ID']
ID_TO_CAMERA = {int(k): v for k, v in mappings['ID_TO_CAMERA'].items()}

print("✅ Loaded label mappings")

In [ ]:
# Define model class (same as training)
class ScriptSegmentationModel(DistilBertPreTrainedModel):
    def __init__(self, config, num_labels=3, num_moods=9, num_cameras=8):
        super().__init__(config)
        self.num_labels = num_labels
        self.num_moods = num_moods
        self.num_cameras = num_cameras
        self.distilbert = DistilBertModel(config)
        self.dropout = nn.Dropout(config.dropout)
        self.beat_classifier = nn.Linear(config.hidden_size, num_labels)
        self.mood_classifier = nn.Sequential(
            nn.Linear(config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_moods)
        )
        self.camera_classifier = nn.Sequential(
            nn.Linear(config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_cameras)
        )
        self.post_init()

    def forward(self, input_ids, attention_mask=None, **kwargs):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        beat_logits = self.beat_classifier(sequence_output)
        pooled_output = sequence_output[:, 0, :]
        mood_logits = self.mood_classifier(pooled_output)
        camera_logits = self.camera_classifier(pooled_output)
        return {'beat_logits': beat_logits, 'mood_logits': mood_logits, 'camera_logits': camera_logits}

In [ ]:
# Load model and tokenizer
print("📥 Loading model...")

config = DistilBertConfig.from_pretrained(MODEL_PATH)
model = ScriptSegmentationModel(config, num_labels=3, num_moods=9, num_cameras=8)

# Load weights
state_dict = torch.load(f"{MODEL_PATH}/model.safetensors", map_location=device)
model.load_state_dict(state_dict, strict=False)
model = model.to(device)
model.eval()

tokenizer = DistilBertTokenizer.from_pretrained(MODEL_PATH)

print("✅ Model loaded successfully")

## 2. Inference Pipeline

In [ ]:
LIGHTING_OPTIONS = {
    'tense': 'low-key dramatic',
    'romantic': 'golden hour',
    'action': 'harsh shadows',
    'mysterious': 'low-key dramatic',
    'comedic': 'high-key bright',
    'dramatic': 'low-key dramatic',
    'serene': 'natural daylight',
    'melancholic': 'moonlit',
    'neutral': 'natural daylight',
}

MUSIC_SUGGESTIONS = {
    'tense': 'ambient electronic tension with low drones',
    'romantic': 'soft piano with string accompaniment',
    'action': 'driving percussion with orchestral hits',
    'mysterious': 'ethereal pads with subtle dissonance',
    'comedic': 'playful woodwinds and pizzicato strings',
    'dramatic': 'orchestral strings building crescendo',
    'serene': 'gentle acoustic guitar and ambient textures',
    'melancholic': 'solo cello with minimal piano',
    'neutral': 'cinematic ambient underscore',
}

In [ ]:
class ScriptSegmenter:
    """Production inference class for script segmentation."""

    def __init__(self, model, tokenizer, device='cpu'):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.scene_pattern = re.compile(
            r'^(INT\.|EXT\.|INT/EXT\.)\s*(.+?)(?:\s*[-–—]\s*(.+))?$',
            re.MULTILINE | re.IGNORECASE
        )
        self.character_pattern = re.compile(r'^([A-Z][A-Z\s\.\'\'\-]+)$', re.MULTILINE)

    def segment_script(self, script: str) -> List[Dict[str, Any]]:
        """Segment a script into beats with metadata."""
        paragraphs = self._split_into_paragraphs(script)

        all_beats = []
        current_beat_text = []
        beat_number = 1

        for para in paragraphs:
            prediction = self._predict_paragraph(para)

            if prediction['is_beat_start'] and current_beat_text:
                beat = self._create_beat(
                    beat_number,
                    '\n'.join(current_beat_text),
                    prediction
                )
                all_beats.append(beat)
                beat_number += 1
                current_beat_text = []

            current_beat_text.append(para)

        # Don't forget last beat
        if current_beat_text:
            prediction = self._predict_paragraph('\n'.join(current_beat_text))
            beat = self._create_beat(
                beat_number,
                '\n'.join(current_beat_text),
                prediction
            )
            all_beats.append(beat)

        # Add music recommendation to first beat
        if all_beats:
            all_beats[0]['music_recommendation'] = MUSIC_SUGGESTIONS.get(
                all_beats[0]['mood'], 'cinematic ambient underscore'
            )

        return all_beats[:8]  # Limit to 8 beats

    def _split_into_paragraphs(self, script: str) -> List[str]:
        """Split script into meaningful paragraphs."""
        lines = script.strip().split('\n')
        paragraphs = []
        current_para = []

        for line in lines:
            stripped = line.strip()
            if not stripped:
                if current_para:
                    paragraphs.append('\n'.join(current_para))
                    current_para = []
            else:
                current_para.append(line)

        if current_para:
            paragraphs.append('\n'.join(current_para))

        return paragraphs

    def _predict_paragraph(self, text: str) -> Dict[str, Any]:
        """Get model predictions for a paragraph."""
        inputs = self.tokenizer(
            text,
            return_tensors='pt',
            truncation=True,
            max_length=256,
            padding='max_length'
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model(**inputs)

        beat_preds = torch.argmax(outputs['beat_logits'], dim=-1)
        mood_pred = torch.argmax(outputs['mood_logits'], dim=-1).item()
        camera_pred = torch.argmax(outputs['camera_logits'], dim=-1).item()

        is_beat_start = (beat_preds[0][1].item() == 1)  # Check position 1

        return {
            'is_beat_start': is_beat_start,
            'mood': ID_TO_MOOD.get(mood_pred, 'neutral'),
            'camera_angle': ID_TO_CAMERA.get(camera_pred, 'medium shot'),
        }

    def _create_beat(self, beat_number: int, text: str, prediction: Dict) -> Dict[str, Any]:
        """Create a beat object with all required fields."""
        characters = self._extract_characters(text)
        visual_desc = self._generate_visual_description(text, characters)
        narrator_line = self._generate_narrator_line(text)
        lighting = LIGHTING_OPTIONS.get(prediction['mood'], 'natural daylight')

        return {
            'beat_number': beat_number,
            'visual_description': visual_desc,
            'camera_angle': prediction['camera_angle'],
            'mood': prediction['mood'],
            'lighting': lighting,
            'characters_present': characters,
            'narrator_line': narrator_line,
        }

    def _extract_characters(self, text: str) -> List[str]:
        """Extract character names from text."""
        matches = self.character_pattern.findall(text)
        stopwords = {'INT', 'EXT', 'THE', 'A', 'AN', 'CUT', 'FADE'}
        return [m.strip() for m in matches if m.strip() not in stopwords][:5]

    def _generate_visual_description(self, text: str, characters: List[str]) -> str:
        """Generate visual description from text."""
        parts = []

        scene_match = self.scene_pattern.search(text)
        if scene_match:
            location = scene_match.group(2).strip()
            time = scene_match.group(3).strip() if scene_match.group(3) else ''
            parts.append(location)
            if time:
                parts.append(time.lower())

        if characters:
            if len(characters) == 1:
                parts.append(f"{characters[0]} in frame")
            else:
                parts.append(f"{', '.join(characters[:2])} in scene")

        action_lines = [
            line.strip() for line in text.split('\n')
            if line.strip() and not line.strip().isupper() and not line.strip().startswith('(')
        ]

        if action_lines:
            parts.append(' '.join(action_lines[:2])[:150])

        return '. '.join(parts) if parts else "Cinematic scene moment"

    def _generate_narrator_line(self, text: str) -> str:
        """Generate a narrator line (100-150 chars)."""
        clean_text = ' '.join(text.split())

        if len(clean_text) <= 150:
            return clean_text

        sentences = clean_text.replace('!', '.').replace('?', '.').split('.')
        sentences = [s.strip() for s in sentences if s.strip()]

        if sentences:
            result = sentences[0]
            if len(result) < 100 and len(sentences) > 1:
                result += '. ' + sentences[1]
            return result[:150]

        return clean_text[:150]

In [ ]:
# Create segmenter instance
segmenter = ScriptSegmenter(model, tokenizer, device)
print("✅ Segmenter ready")

## 3. Test Inference

In [ ]:
# Test with sample script
test_script = """
INT. ABANDONED WAREHOUSE - NIGHT

Rain hammers against broken windows. DETECTIVE MAYA CHEN (40s) steps through the doorway, flashlight cutting through darkness.

MAYA
(whispered)
I know you're here, Marcus.

A SHADOW shifts behind rusted machinery. MARCUS VALE (50s) emerges, hands raised, face half-lit by moonlight.

MARCUS
You shouldn't have come alone.

Maya's hand moves to her holster. Thunder RUMBLES outside.

MAYA
I never do.

RED AND BLUE LIGHTS flood through the windows. Marcus smiles—but it doesn't reach his eyes.
"""

print("🎬 Processing script...\n")
beats = segmenter.segment_script(test_script)

print(f"Generated {len(beats)} beats:\n")
print("="*60)

for beat in beats:
    print(f"\n📍 BEAT {beat['beat_number']}")
    print(f"   🎭 Mood: {beat['mood']}")
    print(f"   📷 Camera: {beat['camera_angle']}")
    print(f"   💡 Lighting: {beat['lighting']}")
    print(f"   👥 Characters: {beat['characters_present']}")
    print(f"   🖼️ Visual: {beat['visual_description'][:80]}...")
    print(f"   🎙️ Narrator: {beat['narrator_line'][:80]}...")
    if 'music_recommendation' in beat:
        print(f"   🎵 Music: {beat['music_recommendation']}")

## 4. Export for Production

In [ ]:
# Export the inference code as a standalone Python file
inference_code = '''
"""PLAYWRIGHT Script Segmentation - Inference Module"""

import re
import json
import torch
import torch.nn as nn
from typing import List, Dict, Any
from transformers import DistilBertTokenizer, DistilBertModel, DistilBertPreTrainedModel, DistilBertConfig

# Label mappings
ID_TO_MOOD = {0: "neutral", 1: "tense", 2: "romantic", 3: "action", 4: "mysterious", 5: "comedic", 6: "dramatic", 7: "serene", 8: "melancholic"}
ID_TO_CAMERA = {0: "medium shot", 1: "close-up", 2: "wide shot", 3: "over-the-shoulder", 4: "pov shot", 5: "tracking shot", 6: "low angle", 7: "high angle"}

LIGHTING_OPTIONS = {
    "tense": "low-key dramatic", "romantic": "golden hour", "action": "harsh shadows",
    "mysterious": "low-key dramatic", "comedic": "high-key bright", "dramatic": "low-key dramatic",
    "serene": "natural daylight", "melancholic": "moonlit", "neutral": "natural daylight",
}

MUSIC_SUGGESTIONS = {
    "tense": "ambient electronic tension", "romantic": "soft piano with strings",
    "action": "driving percussion", "mysterious": "ethereal pads",
    "comedic": "playful woodwinds", "dramatic": "orchestral crescendo",
    "serene": "gentle acoustic guitar", "melancholic": "solo cello",
    "neutral": "cinematic underscore",
}


class ScriptSegmentationModel(DistilBertPreTrainedModel):
    def __init__(self, config, num_labels=3, num_moods=9, num_cameras=8):
        super().__init__(config)
        self.distilbert = DistilBertModel(config)
        self.dropout = nn.Dropout(config.dropout)
        self.beat_classifier = nn.Linear(config.hidden_size, num_labels)
        self.mood_classifier = nn.Sequential(nn.Linear(config.hidden_size, 256), nn.ReLU(), nn.Dropout(0.1), nn.Linear(256, num_moods))
        self.camera_classifier = nn.Sequential(nn.Linear(config.hidden_size, 256), nn.ReLU(), nn.Dropout(0.1), nn.Linear(256, num_cameras))
        self.post_init()

    def forward(self, input_ids, attention_mask=None, **kwargs):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        seq = self.dropout(outputs.last_hidden_state)
        return {"beat_logits": self.beat_classifier(seq), "mood_logits": self.mood_classifier(seq[:, 0, :]), "camera_logits": self.camera_classifier(seq[:, 0, :])}


def load_model(model_path: str, device: str = "cpu"):
    """Load the trained model."""
    config = DistilBertConfig.from_pretrained(model_path)
    model = ScriptSegmentationModel(config)
    model.load_state_dict(torch.load(f"{model_path}/model.safetensors", map_location=device), strict=False)
    model = model.to(device).eval()
    tokenizer = DistilBertTokenizer.from_pretrained(model_path)
    return model, tokenizer


def segment_script(script: str, model, tokenizer, device: str = "cpu") -> List[Dict[str, Any]]:
    """Segment a script into beats."""
    paragraphs = [p for p in script.strip().split("\\n\\n") if p.strip()]
    beats = []

    for i, para in enumerate(paragraphs[:8]):
        inputs = tokenizer(para, return_tensors="pt", truncation=True, max_length=256, padding="max_length")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        mood = ID_TO_MOOD[torch.argmax(outputs["mood_logits"], dim=-1).item()]
        camera = ID_TO_CAMERA[torch.argmax(outputs["camera_logits"], dim=-1).item()]

        beat = {
            "beat_number": i + 1,
            "visual_description": para[:200],
            "camera_angle": camera,
            "mood": mood,
            "lighting": LIGHTING_OPTIONS.get(mood, "natural daylight"),
            "characters_present": re.findall(r"^([A-Z][A-Z\\s]+)$", para, re.MULTILINE)[:3],
            "narrator_line": " ".join(para.split())[:150],
        }

        if i == 0:
            beat["music_recommendation"] = MUSIC_SUGGESTIONS.get(mood, "cinematic underscore")

        beats.append(beat)

    return beats
'''

# Save inference module
with open(f"{MODEL_PATH}/inference.py", 'w') as f:
    f.write(inference_code)

print(f"✅ Inference module saved to: {MODEL_PATH}/inference.py")

In [ ]:
# Create a downloadable zip of the model
import shutil

zip_path = f"{DATA_PATH}/playwright_model"
shutil.make_archive(zip_path, 'zip', MODEL_PATH)

print(f"✅ Model zip created: {zip_path}.zip")
print(f"\n📥 Download from Google Drive: playwright_data/playwright_model.zip")

## 5. API Output Format

In [ ]:
# Show the exact API output format
import json

api_response = {
    "beats": beats,
    "total_beats": len(beats),
    "model_version": "1.0.0"
}

print("📤 API Response Format:")
print(json.dumps(api_response, indent=2))

## Summary

In [ ]:
print("="*50)
print("📊 INFERENCE PIPELINE COMPLETE")
print("="*50)
print(f"\n📁 Files created:")
print(f"  • Model: {MODEL_PATH}/")
print(f"  • Inference module: {MODEL_PATH}/inference.py")
print(f"  • Downloadable zip: {zip_path}.zip")
print(f"\n🚀 To use in your backend:")
print(f"  1. Download playwright_model.zip from Google Drive")
print(f"  2. Extract to your backend/model/ directory")
print(f"  3. Import and use the inference module")
print(f"\n✅ Model ready for production!")

---
## Using the Model in Your Backend

```python
# In your FastAPI backend
from model.inference import load_model, segment_script

# Load once at startup
model, tokenizer = load_model("./model")

# Use in endpoint
@app.post("/api/analyze")
async def analyze(request: ScriptRequest):
    beats = segment_script(request.script, model, tokenizer)
    return {"beats": beats}
```